In [0]:
# Databricks notebook source
import datetime
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "maven_test")
catalog = dbutils.widgets.get("catalog")

# 1. Storage Account & Landing Container
storage_account = "capstoneadls"
landing_container = "landing"

# Ensure Unity Catalog Checkpoint Volume exists (Req 10)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.checkpoints")

# 2. Table mappings
tables_config = {
    "calendar": "operational/calendar/",
    "promotions": "operational/promotions/",
    "regions": "operational/regions/",
    "stores": "operational/stores/",
    "returns": "pos/returns/",
    "transactions": "pos/transactions/"
}

# Unique run batch identifier for auditing (Req 15)
run_batch_id = f"BATCH_{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')}"

print(f"Starting Multi-Table Bronze Ingestion for Catalog: {catalog} | Batch ID: {run_batch_id}")

# 3. Ingestion Loop with Automatic In-Process Restart
for table_name, folder_path in tables_config.items():
    full_table_name = f"{catalog}.bronze.{table_name}"
    source_path = f"abfss://{landing_container}@{storage_account}.dfs.core.windows.net/{folder_path}"
    
    checkpoint_path = f"/Volumes/{catalog}/bronze/checkpoints/{table_name}/state"
    schema_location = f"/Volumes/{catalog}/bronze/checkpoints/{table_name}/schema"
    
    print(f"\nProcessing: {full_table_name} from {source_path}")
    
    max_retries = 3
    for attempt in range(max_retries):
        try:
            # 1. Instantiate readStream inside retry loop to pull updated schema
            # 1. Instantiate readStream inside retry loop to pull updated schema
            df_raw = (
                spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", "csv")
                .option("cloudFiles.schemaLocation", schema_location)
                .option("cloudFiles.inferColumnTypes", "true")
                .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
                .option("cloudFiles.rescuedDataColumn", "_rescued_data")
                .option("header", "true")
                .load(source_path)
                .withColumn("ingestion_timestamp", F.current_timestamp())
                .withColumn("source_file", F.col("_metadata.file_name"))
                .withColumn("source_system", F.lit("ADLS_OPERATIONAL_INGRESS"))
                .withColumn("file_modification_time", F.col("_metadata.file_modification_time"))
                .withColumn("load_batch_id", F.lit(run_batch_id))
            )
            
            # 2. Start writeStream inside retry loop
            query = (
                df_raw.writeStream
                .format("delta")
                .option("checkpointLocation", checkpoint_path)
                .option("mergeSchema", "true")
                .trigger(availableNow=True)
                .toTable(full_table_name)
            )
            
            query.awaitTermination()
            print(f"Successfully loaded data into {full_table_name}")
            break
            
        except Exception as e:
            err_msg = str(e)
            
            # Check for standard Auto Loader schema drift signatures
            is_schema_evolution = any(token in err_msg for token in [
                "UnknownFieldException",
                "UNKNOWN_FIELD_EXCEPTION",
                "NEW_FIELDS_IN_FILE",
                "NEW_FIELDS_IN_RECORD"
            ])
            
            if is_schema_evolution and attempt < max_retries - 1:
                print(f"Schema evolved for {full_table_name}. Restarting stream to absorb new schema (attempt {attempt + 1}/{max_retries})...")
            else:
                raise e

print("\nAll Bronze tables successfully processed.")